In [ ]:
%pip -q install google-genai

In [ ]:
# Configura a API Key do Google Gemini

import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

In [ ]:
!pip install gradio
!pip install pandas
!pip install wordcloud
!pip install matplotlib
!pip install textblob
!pip install altair
!python -m textblob.download_corpora

ERROR: Operation cancelled by user
^C
^C
^C
^C


In [2]:
import gradio as gr
import pandas as pd
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from textblob import TextBlob
import json
import io

ARQUIVO_DICAS = 'dicas.json'

# --- Funções de Persistência de Dados ---
def salvar_dicas(dicas):
    with open(ARQUIVO_DICAS, 'w') as f:
        json.dump(dicas, f)

def carregar_dicas():
    try:
        with open(ARQUIVO_DICAS, 'r') as f:
            return json.load(f)
    except FileNotFoundError:
        return []

# --- Análise de Sentimento ---
def analisar_sentimento(texto):
    try:
        analysis = TextBlob(texto)
        if analysis.sentiment.polarity > 0.1:
            return 'Positivo'
        elif analysis.sentiment.polarity < -0.1:
            return 'Negativo'
        else:
            return 'Neutro'
    except Exception:
        return 'Neutro'

# --- Função Principal para a Interface Gradio ---
def dicas_interface(categoria, recomendacao, descricao, filtro_categoria, filtro_sentimento):
    dicas = carregar_dicas()
    if categoria and recomendacao and descricao:
        dicas.append({'categoria': categoria, 'recomendacao': recomendacao, 'descricao': descricao, 'sentimento': analisar_sentimento(descricao)})
        salvar_dicas(dicas)

    df_dicas = pd.DataFrame(dicas)
    df_filtrado = df_dicas.copy()

    if filtro_categoria != 'Todas' and 'categoria' in df_filtrado.columns:
        df_filtrado = df_filtrado[df_filtrado['categoria'] == filtro_categoria]
    if filtro_sentimento != 'Todas' and 'sentimento' in df_filtrado.columns:
        df_filtrado = df_filtrado[df_filtrado['sentimento'] == filtro_sentimento]

    resultados_html = "<h2>Resultados das Melhores Dicas e Recomendações</h2>"

    if df_filtrado.empty:
        resultados_html += "<p>Nenhuma dica encontrada para os filtros selecionados.</p>"
    else:
        resultados_html += "<h3>Dicas Filtradas</h3>"
        resultados_html += df_filtrado.to_html(index=False)

        resultados_html += "<h3>Análise das Categorias</h3>"
        categoria_counts = df_filtrado['categoria'].value_counts().to_frame(name='Número de Dicas')
        resultados_html += categoria_counts.to_html()

        resultados_html += "<h3>Análise de Sentimentos</h3>"
        sentimento_counts = df_filtrado['sentimento'].value_counts().to_frame(name='Número de Dicas')
        resultados_html += sentimento_counts.to_html()

        # Nuvem de Palavras (gerada como imagem separada)
        texto_combinado = ' '.join(df_filtrado['recomendacao'] + ' ' + df_filtrado['descricao'])
        if texto_combinado:
            wordcloud = WordCloud(width=800, height=400, background_color='white').generate(texto_combinado)
            plt.figure(figsize=(10, 5))
            plt.imshow(wordcloud, interpolation='bilinear')
            plt.axis('off')
            buf = io.BytesIO()
            plt.savefig(buf, format='png')
            buf.seek(0)
            plt.close()
            # Codifica a imagem para exibir diretamente no HTML
            import base64
            image_base64 = base64.b64encode(buf.getvalue()).decode('utf-8')
            resultados_html += "<h3>Nuvem de Palavras das Recomendações</h3>"
            resultados_html += f'<img src="data:image/png;base64,{image_base64}" alt="Nuvem de Palavras">'
        else:
            resultados_html += "<p>Nenhuma palavra para gerar a nuvem.</p>"

    return resultados_html

# --- Criação da Interface Gradio ---
initial_dicas = carregar_dicas()
df_initial = pd.DataFrame(initial_dicas)
categoria_choices = ['Todas'] + sorted(df_initial['categoria'].unique().tolist()) if 'categoria' in df_initial.columns else ['Todas']

iface = gr.Interface(
    fn=dicas_interface,
    inputs=[
        gr.Textbox(label="Categoria da Dica"),
        gr.Textbox(label="Recomendação"),
        gr.Textbox(label="Descrição"),
        gr.Dropdown(choices=categoria_choices, label="Filtrar por Categoria", value="Todas"),
        gr.Dropdown(choices=['Todas', 'Positivo', 'Negativo', 'Neutro'], label="Filtrar por Sentimento", value="Todas")
    ],
    outputs=gr.HTML(label="Resultados"),
    title="Melhores Dicas e Recomendações",
    live=True
)

if __name__ == "__main__":
    iface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9afb214224f8eda238.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
